# Budget Auction — Experiment 1: Public-State Budget Auction

Compares mechanism input variants **A** (GBAM/public-balance), **B** (richer history), **C** (full history) in a Deng-2021-style setting where the GBAM state should be theoretically sufficient.

Run in Google Colab with GPU runtime for best performance.

In [ ]:
%pip install -q git+https://github.com/deepmind/dm-haiku
%pip install -q jaxopt optax flax pyyaml


In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/My\ Research/Dynamic\ Mechanism\ Design/Budget\ Auction
except ImportError:
    pass  # Running locally


In [ ]:
from typing import Dict
import jax
import jax.numpy as jnp
import haiku as hk
import optax
import matplotlib.pyplot as plt
import pickle, json, os

from budget_auction.core import (
    BudgetSimulator, BoundedContPolicy,
    build_bounded_policy_net, plot_loss_dict,
)
from budget_auction.mechanisms.budget_mechanism import BudgetMechanism
from budget_auction.mechanisms.input_adapters import VARIANT_NETWORK
from budget_auction.envs.public_budget_env import create_public_budget_auction
from budget_auction.baselines.static_myopic import run_static_myopic_baseline
from budget_auction.baselines.reserve_budget import run_reserve_budget_baseline
from budget_auction.train.run_acceptance import nested_GD_budget
from budget_auction.train.evaluate import eval_budget_mechanism_multikey, print_eval_report

print('JAX devices:', jax.devices())


## Configuration

In [ ]:
# ---- Select variant and seed ----
VARIANT = 'A'   # 'A' | 'B' | 'C'
SEED    = 42

cfg = dict(
    n_buyers=2, n_goods=1, horizon=5, discount=0.9,
    initial_budget=3.0,
    profit_goal=1.5,
    num_outer_epochs=2000,
    num_inner_steps=300,
    lr_theta=1e-4, lr_w=3e-4,
    num_samples_train=64,
    alpha_weight=1e5, beta_weight=1e3,
    print_iter=100,
    policy_hidden=128, residual_init_std=1.0,
)
print('Config:', cfg)


## Build auction environment

In [ ]:
auction = create_public_budget_auction(
    n_buyers=cfg['n_buyers'], n_goods=cfg['n_goods'],
    horizon=cfg['horizon'],  discount=cfg['discount'],
    initial_budget=cfg['initial_budget'], seed=SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}')
print(f'Initial budgets: {auction.initial_budgets}')


In [ ]:
# Sanity check: sample one initial state
key = jax.random.PRNGKey(0)
s0 = auction.get_initial_state(key)
print('Initial BudgetState:')
print(f'  types  = {s0.current_type}')
print(f'  alpha  = {s0.alpha}  (should be 0)')
print(f'  beta   = {s0.beta}   (should be {auction.initial_budgets})')


## Build mechanism network

In [ ]:
raw_fn = VARIANT_NETWORK[VARIANT]
mechanism_network = hk.without_apply_rng(hk.transform(raw_fn))
policy_network    = build_bounded_policy_net(
    d_priv=auction.d_priv, d_pub=auction.d_pub, G=auction.num_goods,
    hidden=cfg['policy_hidden'], residual_init_std=cfg['residual_init_std'],
)
print(f'Mechanism variant: {VARIANT}')


## Evaluate baselines (no training)

In [ ]:
print('--- Baseline B1: Static Myopic ---')
b1 = run_static_myopic_baseline(auction, num_samples=128, num_keys=8, base_seed=1)
for k, v in b1.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

print('--- Baseline B2: Reserve Budget ---')
b2 = run_reserve_budget_baseline(auction, reserve_multiplier=0.5,
                                  num_samples=128, num_keys=8, base_seed=1)
for k, v in b2.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')


## Train learned mechanism

In [ ]:
logs = nested_GD_budget(
    auction=auction,
    mechanism_network=mechanism_network,
    policy_network=policy_network,
    profit_goal=cfg['profit_goal'],
    num_outer_epochs=cfg['num_outer_epochs'],
    lr_theta=cfg['lr_theta'],
    num_inner_steps=cfg['num_inner_steps'],
    lr_w=cfg['lr_w'],
    num_samples_train=cfg['num_samples_train'],
    alpha_weight=cfg['alpha_weight'],
    beta_weight=cfg['beta_weight'],
    print_iter=cfg['print_iter'],
    seed=SEED,
)
mechanism_params = logs.pop('mechanism_params')
policy_params    = logs.pop('policy_params')
print('Training complete.')


## Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(logs['reg_plus']); axes[0].set_title('Regret+ '+ VARIANT); axes[0].set_xlabel('Outer Epoch')
axes[1].plot(logs['gap_plus']); axes[1].set_title('Profit Gap+');        axes[1].set_xlabel('Outer Epoch')
axes[2].plot(logs['ir']);        axes[2].set_title('IR Loss');            axes[2].set_xlabel('Outer Epoch')
plt.suptitle(f'Exp 1 — Variant {VARIANT}', y=1.02)
plt.tight_layout(); plt.show()


## Evaluate trained mechanism

In [ ]:
eval_keys = jax.random.split(jax.random.PRNGKey(9999), 32)
results = eval_budget_mechanism_multikey(
    auction=auction,
    mechanism_network=mechanism_network,
    mechanism_params=mechanism_params,
    profit_goal=cfg['profit_goal'],
    num_samples=256, keys=eval_keys,
    policy_network=policy_network,
    policy_params=policy_params,
)
print_eval_report(results, prefix=f'Variant {VARIANT}: ')


## Compare A vs B vs C (optional — runs all three)

In [ ]:
all_results = {}
for v in ['A', 'B', 'C']:
    print(f'\n===== Variant {v} =====')
    mnet = hk.without_apply_rng(hk.transform(VARIANT_NETWORK[v]))
    pnet = build_bounded_policy_net(d_priv=auction.d_priv, d_pub=auction.d_pub,
                                    G=auction.num_goods, hidden=128, residual_init_std=1.0)
    vl = nested_GD_budget(auction=auction, mechanism_network=mnet, policy_network=pnet,
                          profit_goal=cfg['profit_goal'],
                          num_outer_epochs=cfg['num_outer_epochs'],
                          lr_theta=cfg['lr_theta'], num_inner_steps=cfg['num_inner_steps'],
                          lr_w=cfg['lr_w'], num_samples_train=cfg['num_samples_train'],
                          alpha_weight=cfg['alpha_weight'], beta_weight=cfg['beta_weight'],
                          print_iter=cfg['print_iter'], seed=SEED)
    vm = vl.pop('mechanism_params'); vp = vl.pop('policy_params')
    vr = eval_budget_mechanism_multikey(auction=auction, mechanism_network=mnet,
                                        mechanism_params=vm, profit_goal=cfg['profit_goal'],
                                        num_samples=256, K=16, base_seed=9999)
    all_results[v] = dict(logs=vl, eval=vr, mech_params=vm, pol_params=vp)

print('\n=== Comparison (Exp 1) ===')
print(f'{"Variant":>8} | {"Revenue":>9} | {"IR Loss":>9} | {"BV Rate":>8}')
print('-' * 45)
for v, r in all_results.items():
    ev = r['eval']
    print(f'{v:>8} | {ev["profit_stats"]["mean"]:>9.4f} | '
          f'{ev["ir_loss_stats"]["mean"]:>9.4f} | '
          f'{ev["budget_viol_stats"]["mean"]:>8.4f}')


## Save outputs

In [ ]:
save_dir = f'outputs/public_budget/seed{SEED}'
os.makedirs(save_dir, exist_ok=True)
with open(f'{save_dir}/logs_{VARIANT}.pkl', 'wb') as f:
    pickle.dump(logs, f)
with open(f'{save_dir}/mech_params_{VARIANT}.pkl', 'wb') as f:
    pickle.dump(mechanism_params, f)
metrics_json = {k: v for k, v in results.items() if isinstance(v, dict)}
metrics_json['variant'] = VARIANT
with open(f'{save_dir}/eval_{VARIANT}.json', 'w') as f:
    json.dump(metrics_json, f, indent=2)
print(f'Saved to {save_dir}')
